In [9]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd
import numpy as np

In [10]:
phones_df = pd.read_csv('datasets/transformed_data/phones.csv')

In [11]:
phones_df.drop(columns=['price']).columns

Index(['brand', 'architecture', 'audio_jack', 'battery_capacity', 'bluetooth',
       'chipset', 'cpu', 'cpu_cores', 'display_type', 'edge',
       'expandable_memory', 'face_unlock', 'features', 'gprs', 'gps', 'gpu',
       'height', 'internal_storage', 'loudspeaker', 'network', 'notch',
       'operating_system', 'os_version', 'pixel_density',
       'primary_camera_resolution', 'quick_charging', 'ram', 'ram_type',
       'refresh_rate', 'resolution', 'screen_protection', 'screen_size',
       'screen_to_body_ratio', 'sim_size', 'sim_slot', 'speed', 'storage_type',
       'thickness', 'touch_screen', 'usb', 'usb_otg', 'usb_type_c', 'video',
       'volte', 'waterproof', 'weight', 'width', 'wlan', 'released_year',
       'released_month', 'released_day'],
      dtype='str')

In [12]:
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics  import  (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [13]:
test_sizes = [.2, .1]

In [14]:
X_data = phones_df.drop(columns=['price'])
y_data = phones_df['price']


def decode(chromosome):

    feature_mask = chromosome[:N_FEATURES].astype(bool)

    n_estimators = int(chromosome[N_FEATURES])
    max_depth = int(chromosome[N_FEATURES + 1])
    min_split = int(chromosome[N_FEATURES + 2])
    min_leaf = int(chromosome[N_FEATURES + 3])
    max_features_gene = int(chromosome[N_FEATURES + 4])

    max_features_map = {
        0: 1.0,
        1:"sqrt",
        2:"log2",
    }

    params = {
        "n_estimators" : n_estimators,
        "max_depth" : max_depth,
        "min_samples_split": min_split,
        "min_samples_leaf": min_leaf,
        "max_features": max_features_map[max_features_gene]
    }


    return feature_mask, params

In [15]:
for i in test_sizes:

    X, X_test, y, y_test = train_test_split(
        X_data,
        y_data,
        test_size=i,
        random_state=42
    )

    N_FEATURES = len(X.columns)

    X = phones_df.drop(columns=['price'])
    y = phones_df['price'] 

    best_params = np.array([0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,1,0,0,0,1,1,0,0,1,1,0,0,0,1,1,0,0,463,11,7,1,1])
    feature_mask, params = decode(best_params)
    selected_features = X.loc[:, feature_mask].columns.to_list()
    selected_X = X.loc[:,feature_mask]

    optimized_model = RandomForestRegressor(
        **params,
        random_state=42
    )

    optimized_model.fit(selected_X, y)

    X_test = X_test.loc[:,feature_mask]
    y_pred = optimized_model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"--- Evaluation Model ---")
    print(f"MAE: {mae}")
    print(f"MSE: {mse}")
    print(f"RMSE: {rmse}")
    print(f"R2 Score: {r2}")

    importance = pd.DataFrame({
        "feature": selected_X.columns,
        "importance": optimized_model.feature_importances_
    })

    importance = importance.sort_values(
        by="importance",
        ascending=False
    )

    print(importance)
    print(f"\n")

--- Evaluation Model ---
MAE: 464240.09804019536
MSE: 886431304177.5013
RMSE: 941504.8083666388
R2 Score: 0.8058895451829565
              feature  importance
3                 ram    0.285786
11               wlan    0.252920
9               video    0.176743
0        display_type    0.100213
12      released_year    0.060903
4   screen_protection    0.038451
5            sim_size    0.038270
8                 usb    0.035136
6            sim_slot    0.011448
7        touch_screen    0.000130
2                gprs    0.000000
1                edge    0.000000
10              volte    0.000000


--- Evaluation Model ---
MAE: 431601.067328039
MSE: 595354496258.446
RMSE: 771592.1826058413
R2 Score: 0.882011096069311
              feature  importance
3                 ram    0.285786
11               wlan    0.252920
9               video    0.176743
0        display_type    0.100213
12      released_year    0.060903
4   screen_protection    0.038451
5            sim_size    0.038270
8   

In [16]:
# import joblib
# import json

# joblib.dump(optimized_model, "models/2nd_phone_price.pkl")

# metadata = {
#     'selected_features': selected_features,
#     'hyperparameter': params
# }

# with open("models/metadata.json", "w") as f:
#     json.dump(metadata, f, indent=4)